In [ ]:
from GalaxySpectrumClassifier.base import Trainable, DatasetProtocol, TrainerProtocol
from GalaxySpectrumClassifier.utils import load_type, resolve_type_kwargs
from skorch import NeuralNetBinaryClassifier, NeuralNetClassifier, NeuralNetRegressor
from pathlib import Path
from typing import Any

In [ ]:
class EpochTrainer(TrainerProtocol):
    def __init__(
        self,
        output_path: str,
        max_epochs,
        model_type: str,
        loss_type: str,
        optimizer_type: str,
        train_dataset_type: str,
        val_dataset_type: str,
        test_dataset_type: str,
        task: str,
        optimizer_kwargs: dict[str, Any] | None = None,
        loss_kwargs: dict[str, Any] | None = None,
        model_args: list[Any] | None = None,
        model_kwargs: dict[str, Any] | None = None,
        calibrator_type: str | None = None,
        calibrator_args: list[Any] | None = None,
        calibrator_kwargs: dict[str, Any] | None = None,
        lr_scheduler_type: str | None = None,
        lr_scheduler_kwargs: dict[str, Any] | None = None,
        callbacks: list[dict[str, str | list[Any] | dict[str, Any]]] | None = None,
        train_dataset_args: list[Any] | None = None,
        train_dataset_kwargs: dict[str, Any] | None = None,
        val_dataset_args: list[Any] | None = None,
        val_dataset_kwargs: dict[str, Any] | None = None,
        test_dataset_args: list[Any] | None = None,
        test_dataset_kwargs: dict[str, Any] | None = None,
        train_loader_kwargs: dict[str, Any] | None = None,
        val_loader_kwargs: dict[str, Any] | None = None,
        test_loader_kwargs: dict[str, Any] | None = None,
        seed: int = 42,
    ):
        self.output_path = Path(output_path).resolve()
        self.output_path.mkdir(parents=True, exist_ok=True)

        # build loss
        criterion_t = load_type(loss_type)
        criterion_kwargs = {}

        for k, kwg in resolve_type_kwargs(loss_kwargs or {}).items():
            criterion_kwargs[f"criterion__{k}"] = kwg

        optim_t = load_type(optimizer_type)
        optim_kwargs = {}
        # add key indicator
        for k, kwg in resolve_type_kwargs(optimizer_kwargs or {}).items():
            optim_kwargs[f"optimizer__{k}"] = kwg

        model_t = load_type(model_type)

        iterator_train_kwargs = {}
        for k, kwg in resolve_type_kwargs(train_loader_kwargs or {}).items():
            iterator_train_kwargs[f"iterator_train__{k}"] = kwg

        iterator_val_kwargs = {}
        for k, kwg in resolve_type_kwargs(val_loader_kwargs or {}).items():
            iterator_val_kwargs[f"iterator_valid__{k}"] = kwg

        # build model

        ## build model type
        skorch_modeltype = None

        if task == "binary-classification":
            skorch_modeltype = NeuralNetBinaryClassifier
        elif task == "multiclass-classification":
            skorch_modeltype = NeuralNetClassifier
        elif task == "regression":
            skorch_modeltype = NeuralNetRegressor
        else:
            raise ValueError(
                "Task unknown, must be one of binary-classification",
                "multiclass-classification",
                "regression",
            )

        module = model_t(
            *(model_args or []), **(resolve_type_kwargs(model_kwargs or {}))
        )

        ## get learning rate scheduler if it's given
        self.model = skorch_modeltype(
            module,
            criterion=criterion_t,
            **criterion_kwargs,
            optimizer=optim_t,
            **optim_kwargs,
        )

        if calibrator_type:
            cal_t = load_type(calibrator_type)
            cal_kwargs = resolve_type_kwargs(calibrator_kwargs or {})
            cal_type()

        # build datasets

        # set data

    def build_model(
        self,
        type: str,
        args: list[Any] | None = None,
        kwargs: dict[str, Any] | None = None,
        calibrator_type: str | None = None,
        calibrator_args: list[Any] | None = None,
        calibrator_kwargs: dict[str, Any] | None = None,
    ) -> Trainable: ...

    def train(
        self,
        train_data: DatasetProtocol,
        validation_data: DatasetProtocol | None = None,
    ) -> Any: ...

    def validate(
        self, data: DatasetProtocol
    ) -> Any: ...  # this perhaps can go, not necessary

    def test(self, data: DatasetProtocol) -> Any: ...

    def save_snapshot(self, path: str) -> None: ...

    @classmethod
    def load_snapshot(cls, path: str) -> "TrainerProtocol": ...

    def save_model(self, path: str) -> None: ...

    @staticmethod
    def load_model(path: str) -> Trainable: ...